In [ ]:
# ============================================
# SECIS Crisis Management - RL Training with Live Backend
# SSL Error Handling Version (Adaptive Agent Training + Comparison)
# ============================================

# 📦 Install Dependencies
!pip install requests matplotlib numpy pandas urllib3

# ============================================
# 🔧 CONFIGURATION
# ============================================
import requests
import random
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import urllib3
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Disable SSL warnings (ngrok free tier)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Your Hugging Face Space URL (Publicly deployed backend)
BASE_URL = "https://exterior-trial-squeezing.ngrok-free.dev"

# API Endpoints
RESET_URL = BASE_URL + "/api/reset"
STEP_URL = BASE_URL + "/api/step"
STATE_URL = BASE_URL + "/api/state"
TELEMETRY_URL = BASE_URL + "/api/telemetry"
LEADERBOARD_URL = BASE_URL + "/api/leaderboard"
REFLECTION_URL = BASE_URL + "/api/reflection"
CONTROL_PARAMS_URL = BASE_URL + "/api/control-parameters"

print(f"🚀 Backend URL: {BASE_URL}")
print("✅ Using Hugging Face Space - Publicly accessible 24/7")
print("💡 No need for ngrok tunnel")

# ============================================
# 🔧 ROBUST REQUESTS WITH RETRY LOGIC (IMPROVED)
# ============================================
def create_session():
    """Create a requests session with retry logic"""
    session = requests.Session()

    # Retry strategy
    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS", "POST"]
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    return session

# Create session
session = create_session()

def safe_request(method, url, **kwargs):
    """Make a safe request with SSL error handling (IMPROVED)"""
    kwargs.setdefault('verify', False)  # Disable SSL verification for ngrok
    kwargs.setdefault('timeout', 60)   # Increased timeout from 30 to 60 seconds

    max_retries = 5  # Increased from 3 to 5
    for attempt in range(max_retries):
        try:
            if method == 'GET':
                response = session.get(url, **kwargs)
            elif method == 'POST':
                response = session.post(url, **kwargs)
            elif method == 'DELETE':
                response = session.delete(url, **kwargs)
            elif method == 'PUT':
                response = session.put(url, **kwargs)
            else:
                raise ValueError(f"Unknown method: {method}")

            return response

        except (requests.exceptions.SSLError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt < max_retries - 1:
                print(f"⚠️ Connection Error (attempt {attempt + 1}/{max_retries}): {type(e).__name__}")
                print(f"   Retrying in {3 ** attempt} seconds...")
                time.sleep(3 ** attempt)  # Increased backoff from 2 to 3
            else:
                print(f"❌ Failed after {max_retries} attempts")
                raise
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"⚠️ Request Error (attempt {attempt + 1}/{max_retries}): {e}")
                print(f"   Retrying in {3 ** attempt} seconds...")
                time.sleep(3 ** attempt)
            else:
                raise

# ============================================
# 🔥 CHECKPOINT 1 — Test Connection
# ============================================
print("\n" + "="*50)
print("🔥 CHECKPOINT 1: Test Connection")
print("="*50)
try:
    r = safe_request('GET', BASE_URL)
    print(f"Status Code: {r.status_code}")
    print(f"Response: {r.text[:200]}")
    assert r.status_code == 200, "Connection test failed!"
    print("✅ Connection test passed!")
except Exception as e:
    print(f"❌ Connection test failed: {e}")
    raise

# ============================================
# 🔥 CHECKPOINT 2 — Get State
# ============================================
print("\n" + "="*50)
print("🔥 CHECKPOINT 2: Get State")
print("="*50)
try:
    r = safe_request('GET', STATE_URL)
    state = r.json()
    print(f"State keys: {state.keys()}")
    print(f"✅ State retrieved successfully!")
except Exception as e:
    print(f"❌ State check failed: {e}")
    raise

# ============================================
# 🔥 CHECKPOINT 3 — Reset Environment
# ============================================
print("\n" + "="*50)
print("🔥 CHECKPOINT 3: Reset Environment")
print("="*50)
try:
    r = safe_request('POST', RESET_URL)
    print(f"Status Code: {r.status_code}")
    assert r.status_code == 200, "Reset failed!"
    print("✅ Environment reset successful!")
except Exception as e:
    print(f"❌ Reset failed: {e}")
    raise

# ============================================
# 🧠 AGENT TRAINING CONFIGURATION (FIXED API FORMAT)
# ============================================
NUM_EPISODES = 40  # Train adaptive agent for 40 episodes
COMPARISON_EPISODES = 10  # Run 5 episodes each for greedy and conservative for comparison
MAX_STEPS = 30     # Reduced to 30 for faster episodes

# Primary agent to train
PRIMARY_AGENT = "adaptive"
# Comparison agents
COMPARISON_AGENTS = ["greedy", "conservative"]

print(f"\n📊 Training Configuration (ADAPTIVE AGENT TRAINING + COMPARISON):")
print(f"   Primary Agent: {PRIMARY_AGENT} ({NUM_EPISODES} episodes)")
print(f"   Comparison Agents: {COMPARISON_AGENTS} ({COMPARISON_EPISODES} episodes each)")
print(f"   Max Steps: {MAX_STEPS}")
print(f"   Estimated Time: ~{(NUM_EPISODES + len(COMPARISON_AGENTS) * COMPARISON_EPISODES) * 0.5:.0f}-{(NUM_EPISODES + len(COMPARISON_AGENTS) * COMPARISON_EPISODES) * 1:.0f} minutes")
print(f"   ⚠️ NOTE: Backend uses built-in agents, not custom Q-learning")
print(f"   This notebook trains adaptive agent and compares with greedy/conservative")

episode_rewards = []
cumulative_rewards = []
steps_per_episode = []
agent_history = []

def step_env(agent_type):
    """Execute step with specified backend agent"""
    request_data = {
        "agent_type": agent_type,
        "multi_agent": False
    }
    r = safe_request('POST', STEP_URL, json=request_data)
    data = r.json()

    # Handle both single-agent and multi-agent response formats
    if "multi_agent" in data and data["multi_agent"]:
        # Multi-agent response
        agent_data = data["agents"].get(agent_type, {})
        return (
            agent_data.get("state", {}),
            agent_data.get("reward", 0),
            data.get("done", False),
            agent_data.get("metadata", {})
        )
    else:
        # Single-agent response
        return (
            data.get("state", {}),
            data.get("reward", 0),
            data.get("done", False),
            data.get("metadata", {})
        )

# ============================================
# 🎯 TRAINING LOOP (ADAPTIVE AGENT + COMPARISON)
# ============================================
print("\n" + "="*50)
print("🚀 TRAINING STARTED")
print("="*50)
print(f"Phase 1: Train {PRIMARY_AGENT} agent ({NUM_EPISODES} episodes)")
print(f"Phase 2: Compare with {COMPARISON_AGENTS} ({COMPARISON_EPISODES} episodes each)")
print("="*50)

start_time = time.time()

# Phase 1: Train adaptive agent
print("\n" + "="*50)
print(f"📈 PHASE 1: Training {PRIMARY_AGENT.upper()} Agent")
print("="*50)

for episode in range(NUM_EPISODES):
    # Reset environment
    safe_request('POST', RESET_URL)

    # Use primary agent
    agent_type = PRIMARY_AGENT
    agent_history.append(agent_type)

    total_reward = 0
    done = False
    steps = 0

    while not done and steps < MAX_STEPS:
        try:
            state, reward, done, metadata = step_env(agent_type)
            total_reward += reward
            steps += 1
        except Exception as e:
            print(f"⚠️ Step error at episode {episode+1}, step {steps}: {e}")
            # Skip this step and continue
            done = False

    episode_rewards.append(total_reward)
    cumulative = (cumulative_rewards[-1] if cumulative_rewards else 0) + total_reward
    cumulative_rewards.append(cumulative)
    steps_per_episode.append(steps)

    # Print progress every 5 episodes
    if (episode + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Episode {episode+1}/{NUM_EPISODES} | Agent: {agent_type} | Reward: {total_reward:.2f} | Steps: {steps} | Time: {elapsed:.1f}s")

# Phase 2: Comparison with greedy and conservative
print("\n" + "="*50)
print(f"📊 PHASE 2: Comparison with {COMPARISON_AGENTS}")
print("="*50)

for comp_agent in COMPARISON_AGENTS:
    print(f"\n--- Training {comp_agent.upper()} for comparison ---")
    for episode in range(COMPARISON_EPISODES):
        # Reset environment
        safe_request('POST', RESET_URL)

        agent_type = comp_agent
        agent_history.append(agent_type)

        total_reward = 0
        done = False
        steps = 0

        while not done and steps < MAX_STEPS:
            try:
                state, reward, done, metadata = step_env(agent_type)
                total_reward += reward
                steps += 1
            except Exception as e:
                print(f"⚠️ Step error at episode {episode+1}, step {steps}: {e}")
                # Skip this step and continue
                done = False

        episode_rewards.append(total_reward)
        cumulative = (cumulative_rewards[-1] if cumulative_rewards else 0) + total_reward
        cumulative_rewards.append(cumulative)
        steps_per_episode.append(steps)

        # Print progress
        elapsed = time.time() - start_time
        print(f"Episode {episode+1}/{COMPARISON_EPISODES} | Agent: {agent_type} | Reward: {total_reward:.2f} | Steps: {steps} | Time: {elapsed:.1f}s")

training_time = time.time() - start_time

print("\n" + "="*50)
print("✅ TRAINING COMPLETED")
print("="*50)
print(f"Total Training Time: {training_time:.2f}s ({training_time/60:.1f} minutes)")
print(f"Average Reward: {np.mean(episode_rewards):.2f}")
print(f"Best Episode Reward: {np.max(episode_rewards):.2f}")
print(f"Worst Episode Reward: {np.min(episode_rewards):.2f}")
print("="*50)

# ============================================
# 📊 VISUALIZATION
# ============================================
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Simple Moving Average (Learning Trend) - adaptive agent only
adaptive_rewards = episode_rewards[:NUM_EPISODES]
window = min(8, len(adaptive_rewards))
if len(adaptive_rewards) >= window:
    moving_avg = pd.Series(adaptive_rewards).rolling(window=window).mean()
    axes[0, 0].plot(moving_avg, linewidth=2, label=f'Moving Average (window={window})', marker='D', markersize=6, color='blue')
    axes[0, 0].set_title('Learning Trend (Adaptive Agent - Simple MA)', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# Cumulative Rewards
axes[0, 1].plot(cumulative_rewards, marker='*', markersize=8, linewidth=2, color='green')
axes[0, 1].set_title('Cumulative Rewards (All Agents)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Cumulative Reward')
axes[0, 1].grid(True, alpha=0.3)

# Learning Curve Comparison (All Agents)
agent_names = [PRIMARY_AGENT] + COMPARISON_AGENTS
colors = {'adaptive': 'blue', 'greedy': 'orange', 'conservative': 'red'}
window = min(8, len(episode_rewards))

for agent in agent_names:
    agent_eps = [r for a, r in zip(agent_history, episode_rewards) if a == agent]
    if agent_eps:
        moving_avg = pd.Series(agent_eps).rolling(window=window).mean()
        axes[1, 0].plot(range(len(agent_eps)), moving_avg, linewidth=2, label=f'{agent.capitalize()}', marker='o', markersize=5, color=colors[agent])

axes[1, 0].set_title('Learning Curve Comparison (All Agents)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Reward (Moving Average)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Hide the empty subplot
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# ============================================
# 📋 SUMMARY
# ============================================
print("\n" + "="*50)
print("📋 TRAINING SUMMARY (ADAPTIVE AGENT + COMPARISON)")
print("="*50)
print(f"Total Episodes: {len(episode_rewards)}")
print(f"  - {PRIMARY_AGENT}: {NUM_EPISODES} episodes")
print(f"  - {COMPARISON_AGENTS[0]}: {COMPARISON_EPISODES} episodes")
print(f"  - {COMPARISON_AGENTS[1]}: {COMPARISON_EPISODES} episodes")
print(f"Training Time: {training_time:.2f}s ({training_time/60:.1f} minutes)")
print(f"Average Reward: {np.mean(episode_rewards):.2f}")
print(f"Best Episode Reward: {np.max(episode_rewards):.2f}")
print(f"Worst Episode Reward: {np.min(episode_rewards):.2f}")
print(f"\nAgent Performance:")
for agent in [PRIMARY_AGENT] + COMPARISON_AGENTS:
    agent_eps = [r for a, r in zip(agent_history, episode_rewards) if a == agent]
    if agent_eps:
        print(f"   {agent}: {np.mean(agent_eps):.2f} avg reward ({len(agent_eps)} episodes)")
print("="*50)

print("\n✅ All done! Training and comparison complete.")
print("\n💡 Important Notes:")
print(f"   - Primary agent trained: {PRIMARY_AGENT} ({NUM_EPISODES} episodes)")
print(f"   - Comparison agents: {COMPARISON_AGENTS} ({COMPARISON_EPISODES} episodes each)")
print("   - Backend: Hugging Face Space (publicly accessible 24/7)")
print("   - Learning trend uses Simple Moving Average with window=8")
print("   - Smaller window = more responsive to recent changes")
print("   - Learning curve comparison shows adaptive vs baseline performance")
print("   - Cumulative rewards graph shows all agents")
print("   - Backend uses built-in agents, not custom Q-learning")
print("   - For custom RL training, you would need to modify the backend")
print("   - Or use the environment directly without the HTTP API")